✈️ AirFly Insights – Final Combined Notebook

In [ ]:
#Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# For maps
import plotly.express as px

sns.set_style("whitegrid")

In [ ]:
#Load Dataset (ONLY ONCE)

df = pd.read_csv('data/Flight_Dataset.csv')

print("Initial Shape:", df.shape)
df.head()

In [ ]:
#Data Understanding

pd.set_option('display.max_columns', 50)

print("\nColumns:\n", df.columns)

print("\nData Info:")
df.info()

print("\nSummary Statistics:")
df.describe()

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nMissing Values:\n")
print(df.isnull().sum())

In [ ]:
#Investigating NULL Values

print("\nSample rows where DEP_TIME is NULL:")
print(df[df['DEP_TIME'].isnull()].head())

In [ ]:
#Handling Cancellation Code

df['CANCELLATION_CODE'] = df['CANCELLATION_CODE'].fillna('NC')

In [ ]:
#Removing Invalid CRS_ELAPSED_TIME

df.dropna(subset=['CRS_ELAPSED_TIME'], inplace=True)

In [ ]:
#Delay Handling Logic

delays = [
    'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
    'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
    'DELAY_DUE_LATE_AIRCRAFT'
]

mask = (
    (df['CANCELLATION_CODE'] == 'NC') &
    (df['DIVERTED'] == 0) &
    (df['ARR_DELAY'] <= 0)
)

df.loc[mask, delays] = df.loc[mask, delays].fillna(0.0)

In [ ]:
#Operational NULL Analysis

print("\nDEP_TIME NULL Analysis:")
print(df[df['DEP_TIME'].isna()][['CANCELLED','DIVERTED']].value_counts())

print("\nTAXI_IN NULL Analysis:")
print(df[df['TAXI_IN'].isna()][['CANCELLED','DIVERTED']].value_counts())

In [ ]:
#Removing Logical Errors

invalid_rows = df[
    (df['TAXI_IN'].isna()) &
    (df['CANCELLED'] == 0) &
    (df['DIVERTED'] == 0)
]

print("\nInvalid Rows Removed:", invalid_rows.shape)

df = df.drop(invalid_rows.index)

In [ ]:
#Handling Minor Delays

mask1 = (
    (df['CANCELLED'] == 0) &
    (df['DIVERTED'] == 0) &
    (df['ARR_DELAY'] <= 15)
)

df.loc[mask1, delays] = df.loc[mask1, delays].fillna(0.0)

In [ ]:
#Final Missing Check

print("\nRemaining Null Values:\n")
print(df.isnull().sum())

In [ ]:
#Extra Validation

print("\nFlights with Negative Arrival Delay:")
print(df[df['ARR_DELAY'] < 0].shape)

print("\nFlights with Extreme Delay (>1000 mins):")
print(df[df['ARR_DELAY'] > 1000].shape)

In [ ]:
#Data Type Conversion

df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

df['CANCELLED'] = df['CANCELLED'].astype('int8')
df['DIVERTED'] = df['DIVERTED'].astype('int8')

cat_cols = [
    'AIRLINE','AIRLINE_DOT','AIRLINE_CODE',
    'ORIGIN','ORIGIN_CITY','DEST','DEST_CITY',
    'CANCELLATION_CODE'
]

for col in cat_cols:
    df[col] = df[col].astype('category')

In [ ]:
#Feature Engineering

df['YEAR'] = df['FL_DATE'].dt.year
df['MONTH'] = df['FL_DATE'].dt.month
df['MONTH_NAME'] = df['FL_DATE'].dt.month_name()
df['WEEKDAY'] = df['FL_DATE'].dt.day_name()

df['DEP_HOUR'] = df['DEP_TIME'] // 100
df['DEP_HOUR'] = df['DEP_HOUR'].fillna(df['CRS_DEP_TIME'] // 100)

df['ROUTE'] = df['ORIGIN'].astype(str) + "-" + df['DEST'].astype(str)

In [ ]:
#KPI Summary

total_flights = len(df)
unique_airlines = df['AIRLINE'].nunique()

cancelled_pct = (df['CANCELLED'].sum() / total_flights) * 100
diverted_pct = (df['DIVERTED'].sum() / total_flights) * 100

print("\n===== KPI SUMMARY =====")
print("Total Flights:", total_flights)
print("Unique Airlines:", unique_airlines)
print(f"Cancelled Flights: {cancelled_pct:.2f}%")
print(f"Diverted Flights: {diverted_pct:.2f}%")

In [ ]:
#Active Flights

active_flights = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)]

In [ ]:
#Cancellation Analysis

df['CANCELLED'].value_counts().plot.pie(
    labels=['Not Cancelled','Cancelled'],
    autopct='%1.1f%%'
)
plt.title("Flight Cancellation Percentage")
plt.show()

# Insight: Most flights are not cancelled

In [ ]:
#Top Airlines

top_airlines = active_flights['AIRLINE'].value_counts().head(5)

sns.barplot(x=top_airlines.values, y=top_airlines.index)
plt.title("Top Airlines")
plt.show()

# Insight: Few airlines dominate operations

In [ ]:
#Flights by Month

month_counts = active_flights['MONTH_NAME'].value_counts()

sns.barplot(x=month_counts.index, y=month_counts.values)
plt.xticks(rotation=45)
plt.show()

# Insight: Travel demand varies seasonally

In [ ]:
#Delay Distribution

delayed = active_flights[active_flights['ARR_DELAY'] > 0]

sns.histplot(delayed['ARR_DELAY'], bins=50, kde=True)
plt.show()

# Insight: Most delays are small

In [ ]:
#ADVANCED VISUALIZATION

#Top Routes

top_routes = active_flights['ROUTE'].value_counts().head(10)

sns.barplot(x=top_routes.values, y=top_routes.index)
plt.show()

# Insight: High traffic routes exist

In [ ]:
#Route Delay

route_delay = active_flights.groupby('ROUTE')['ARR_DELAY'].mean().nlargest(10)

sns.barplot(x=route_delay.values, y=route_delay.index)
plt.show()

# Insight: Some routes consistently delayed

In [ ]:
#Heatmap

heatmap_data = active_flights.pivot_table(
    values='ARR_DELAY',
    index='ORIGIN',
    columns='DEST',
    aggfunc='mean'
).iloc[:20, :20]

sns.heatmap(heatmap_data, cmap="coolwarm")
plt.show()

# Insight: Certain airport pairs are delay-prone

In [ ]:
#Map Visualization

map_data = active_flights.groupby(['ORIGIN_CITY']).size().reset_index(name='FLIGHTS')

fig = px.scatter_geo(
    map_data,
    size="FLIGHTS",
    hover_name="ORIGIN_CITY",
    title="Flights by City"
)

fig.show()

In [ ]:
#ADVANCED ANALYTICS

#Correlation Heatmap

numeric_cols = active_flights.select_dtypes(include=np.number)

sns.heatmap(numeric_cols.corr(), cmap="coolwarm")
plt.show()

# Insight: Delay variables strongly correlated

In [ ]:
#Delay vs Distance

sns.scatterplot(x='DISTANCE', y='ARR_DELAY', data=active_flights)
plt.show()

# Insight: Distance has weak impact

In [ ]:
#Delay by Hour

hour_delay = active_flights.groupby('DEP_HOUR')['ARR_DELAY'].mean()

sns.lineplot(x=hour_delay.index, y=hour_delay.values)
plt.show()

# Insight: Evening flights show higher delays

In [ ]:
#Airline Performance

performance = active_flights.groupby('AIRLINE').agg({
    'ARR_DELAY':'mean',
    'FL_NUMBER':'count'
})

performance['SCORE'] = performance['FL_NUMBER'] / (performance['ARR_DELAY'] + 1)

top_perf = performance.sort_values(by='SCORE', ascending=False).head(10)

sns.barplot(x=top_perf['SCORE'], y=top_perf.index)
plt.show()

# Insight: High volume + low delay = best airlines

In [ ]:
#Final KPI Dashboard

print("===== FINAL KPI =====")
print("Total Flights:", len(df))
print("Cancellation Rate:", df['CANCELLED'].mean()*100)
print("Diversion Rate:", df['DIVERTED'].mean()*100)
print("Avg Delay:", active_flights['ARR_DELAY'].mean())

In [ ]:
#FINAL INSIGHTS

"""
• Most flights operate successfully
• Carrier delays are major contributors
• Evening flights face higher delays
• Distance has minimal effect
• Certain routes and airports show repeated delays
"""

In [ ]:
#BUSINESS RECOMMENDATIONS

"""
• Optimize evening scheduling
• Improve aircraft turnaround
• Focus on high-delay routes
• Enhance airport efficiency
"""